In [1]:
import requests
from datetime import datetime
import pymysql

In [2]:
API_key = "ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki"
url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&api_key={API_key}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    print("Error:", response.status_code)

In [3]:
data

{'links': {'next': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-07&end_date=2024-01-13&detailed=false&api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki',
  'previous': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2023-12-26&end_date=2024-01-01&detailed=false&api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki',
  'self': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&detailed=false&api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki'},
 'element_count': 31,
 'near_earth_objects': {'2024-01-02': [{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/2415949?api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki'},
    'id': '2415949',
    'neo_reference_id': '2415949',
    'name': '415949 (2001 XY10)',
    'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2415949',
    'absolute_magnitude_h': 19.37,
    'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.3552670883,
      'estimated_diameter_max': 0.7944013596},
 

In [4]:
data.keys()

dict_keys(['links', 'element_count', 'near_earth_objects'])

In [5]:
data['links']

{'next': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-07&end_date=2024-01-13&detailed=false&api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki',
 'previous': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2023-12-26&end_date=2024-01-01&detailed=false&api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki',
 'self': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&detailed=false&api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki'}

In [6]:
data['element_count']

31

In [7]:
data['near_earth_objects']

{'2024-01-02': [{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/2415949?api_key=ZXjUgAada78FN12bxgvxcrb0e5jJeU5H89A3G5ki'},
   'id': '2415949',
   'neo_reference_id': '2415949',
   'name': '415949 (2001 XY10)',
   'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2415949',
   'absolute_magnitude_h': 19.37,
   'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.3552670883,
     'estimated_diameter_max': 0.7944013596},
    'meters': {'estimated_diameter_min': 355.267088298,
     'estimated_diameter_max': 794.4013596028},
    'miles': {'estimated_diameter_min': 0.2207526659,
     'estimated_diameter_max': 0.4936179672},
    'feet': {'estimated_diameter_min': 1165.5744739718,
     'estimated_diameter_max': 2606.3037566394}},
   'is_potentially_hazardous_asteroid': False,
   'close_approach_data': [{'close_approach_date': '2024-01-02',
     'close_approach_date_full': '2024-Jan-02 13:14',
     'epoch_date_close_approach': 1704201240000,
     'rela

In [8]:
type(data['near_earth_objects'])

dict

In [9]:
data['near_earth_objects'].keys()

dict_keys(['2024-01-02', '2024-01-01', '2024-01-04', '2024-01-03', '2024-01-06', '2024-01-05', '2024-01-07'])

In [10]:
asteroids_data = []
target = 10000

while len(asteroids_data) < target:
    response = requests.get(url)
    data = response.json()
    details = data['near_earth_objects']

    for date, ast_details in details.items():
        for ast in ast_details:
            # Check if 'close_approach_data' is not empty
            if ast['close_approach_data']:
                close_data = ast['close_approach_data'][0]  # It's a list of dicts
                asteroids_data.append(dict(
                    id=int(ast['id']),
                    neo_reference_id=int(ast['neo_reference_id']),
                    name=ast['name'],
                    absolute_magnitude_h=ast['absolute_magnitude_h'],
                    dia_min=ast['estimated_diameter']['kilometers']['estimated_diameter_min'],
                    dia_max=ast['estimated_diameter']['kilometers']['estimated_diameter_max'],
                    hazardous_ast=ast['is_potentially_hazardous_asteroid'],
                    approach_date=close_data['close_approach_date'],
                    relative_velocity_kmph=close_data['relative_velocity']['kilometers_per_hour'],
                    astronomical=close_data['miss_distance']['astronomical'],
                    miss_distance_km=close_data['miss_distance']['kilometers'],
                    miss_distance_lunar=close_data['miss_distance']['lunar'],
                    orbiting_body=close_data['orbiting_body']
                ))

            if len(asteroids_data) >= target:
                break
        if len(asteroids_data) >= target:
            break

    # Move to next page
    if 'next' in data['links']:
        url = data['links']['next']
    else:
        break  # Exit if no more pages



In [11]:
asteroids_data

[{'id': 2415949,
  'neo_reference_id': 2415949,
  'name': '415949 (2001 XY10)',
  'absolute_magnitude_h': 19.37,
  'dia_min': 0.3552670883,
  'dia_max': 0.7944013596,
  'hazardous_ast': False,
  'approach_date': '2024-01-02',
  'relative_velocity_kmph': '57205.8951204341',
  'astronomical': '0.3372535274',
  'miss_distance_km': '50452409.349026638',
  'miss_distance_lunar': '131.1916221586',
  'orbiting_body': 'Earth'},
 {'id': 3160747,
  'neo_reference_id': 3160747,
  'name': '(2003 SR84)',
  'absolute_magnitude_h': 26.0,
  'dia_min': 0.0167708462,
  'dia_max': 0.0375007522,
  'hazardous_ast': False,
  'approach_date': '2024-01-02',
  'relative_velocity_kmph': '38589.054833182',
  'astronomical': '0.1323425924',
  'miss_distance_km': '19798169.933318188',
  'miss_distance_lunar': '51.4812684436',
  'orbiting_body': 'Earth'},
 {'id': 3309828,
  'neo_reference_id': 3309828,
  'name': '(2005 YQ96)',
  'absolute_magnitude_h': 20.62,
  'dia_min': 0.1997813652,
  'dia_max': 0.4467247133,
  

In [12]:
len(asteroids_data)

10000

In [13]:
# MySQL connection
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="loganWolf@123"
)

cursor = conn.cursor()

# Create database
cursor.execute("CREATE DATABASE IF NOT EXISTS nasa_asteroids_db")
print("MySQL database 'nasa_asteroids_db' created successfully")

MySQL database 'nasa_asteroids_db' created successfully


In [14]:
# Use database
cursor.execute("USE nasa_asteroids_db")

0

In [15]:
asteroids_table = """
CREATE TABLE IF NOT EXISTS asteroids(
    id INT,
    name VARCHAR(255),
    absolute_magnitude_h FLOAT,
    estimated_diameter_min_km FLOAT,
    estimated_diameter_max_km FLOAT,
    is_potentially_hazardous_asteroid BOOLEAN
)
"""

cursor.execute(asteroids_table)

0

In [16]:
close_table = """
CREATE TABLE IF NOT EXISTS close_approach(
    neo_reference_id INT,
    close_approach_date DATE,
    relative_velocity_kmph FLOAT,
    astronomical FLOAT,
    miss_distance_km FLOAT,
    miss_distance_lunar FLOAT,
    orbiting_body VARCHAR(50)
)
"""

cursor.execute(close_table)


0

In [17]:
for ast in asteroids_data:
    try:
        asteroid_id = int(ast['id'])
        name = ast['name']
        magnitude = ast['absolute_magnitude_h']

        dia_min = ast['dia_min']
        dia_max = ast['dia_max']

        hazardous = ast['hazardous_ast']

    except KeyError:
        continue

    cursor.execute("""
    INSERT INTO asteroids
    (id, name, absolute_magnitude_h, estimated_diameter_min_km,
     estimated_diameter_max_km, is_potentially_hazardous_asteroid)
    VALUES (%s, %s, %s, %s, %s, %s)
    """, (asteroid_id, name, magnitude, dia_min, dia_max, hazardous))


    # Insert close approach data (already flattened)
    try:
        approach_date = ast['approach_date']
        velocity = ast['relative_velocity_kmph']
        astronomical = ast['astronomical']
        miss_km = ast['miss_distance_km']
        miss_lunar = ast['miss_distance_lunar']
        orbiting = ast['orbiting_body']

    except KeyError:
        continue

    cursor.execute("""
    INSERT INTO close_approach
    (neo_reference_id, close_approach_date, relative_velocity_kmph,
     astronomical, miss_distance_km, miss_distance_lunar, orbiting_body)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (asteroid_id, approach_date, velocity, astronomical, miss_km, miss_lunar, orbiting))

conn.commit()
cursor.close()
conn.close()